# 📚 Olasılıksal Dil Modelleri: From Zero to Hero

Bu notebook, Doğal Dil İşleme (NLP) alanında temel olan olasılıksal dil modellerini kapsamlı bir şekilde ele almaktadır. Hidden Markov Models (HMM), Maximum Entropy Models ve N-gram modelleri konularında teoriden pratiğe kusursuz bir yolculuk sunuyoruz.

## 🎯 Öğrenme Hedefleri
- Olasılıksal dil modellerinin temel kavramlarını anlama
- Hidden Markov Models ile POS tagging
- Maximum Entropy Models ile duygu analizi
- N-gram modelleri ile dil modelleme
- Gerçek veri setleri üzerinde pratik uygulamalar
- Profesyonel görselleştirme teknikleri

---

**Yazar:** NLP Eğitim Serisi  
**Tarih:** 2025  
**Seviye:** Başlangıç → İleri Düzey

## 📋 İçindekiler

1. [📚 Kütüphaneler ve Veri Hazırlığı](#kutuphaneler)
2. [🎲 Olasılıksal Dil Modellerine Giriş](#giris)
3. [🔗 Hidden Markov Models (HMM)](#hmm)
   - 3.1 [Temel Kavramlar](#hmm-temel)
   - 3.2 [POS Tagging ile Uygulama](#hmm-pos)
   - 3.3 [Gelişmiş HMM Uygulamaları](#hmm-gelismis)
4. [⚖️ Maximum Entropy Models](#maxent)
   - 4.1 [Teori ve Matematiksel Temeller](#maxent-teori)
   - 4.2 [Duygu Analizi Uygulaması](#maxent-duygu)
   - 4.3 [Metin Sınıflandırma](#maxent-siniflandirma)
5. [📊 N-gram Dil Modelleri](#ngram)
   - 5.1 [Unigram, Bigram, Trigram](#ngram-temel)
   - 5.2 [Dil Modelleme ve Metin Üretimi](#ngram-uretim)
   - 5.3 [Smoothing Teknikleri](#ngram-smoothing)
6. [🚀 5 Kapsamlı Örnek Uygulama](#ornekler)
7. [📈 Sonuçlar ve Değerlendirme](#sonuc)

# 📚 Kütüphaneler ve Veri Hazırlığı {#kutuphaneler}

Bu bölümde projemiz için gerekli olan tüm kütüphaneleri içe aktaracak ve veri setlerimizi hazırlayacağız.

In [ ]:
# Temel kütüphaneleri içe aktarma
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import random
import re
import warnings
warnings.filterwarnings('ignore')

# NLP kütüphaneleri
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, brown, movie_reviews, reuters, conll2000
from nltk.util import ngrams
from nltk.tag import hmm
from nltk.classify import MaxentClassifier, accuracy
from nltk.probability import FreqDist, ConditionalFreqDist

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

# Görselleştirme için pastel renk paleti
pastel_colors = ['#FFB3BA', '#BAFFC9', '#BAE1FF', '#FFFFBA', '#FFD4BA', '#E1BAFF', '#C9FFE1', '#FFE1BA']
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette(pastel_colors)

# Grafik ayarları
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14

print("✅ Tüm kütüphaneler başarıyla yüklendi!")
print("🎨 Pastel renk paleti ayarlandı")
print("📊 Grafik ayarları yapılandırıldı")

In [ ]:
# NLTK veri setlerini indirme
nltk_downloads = [
    'punkt', 'stopwords', 'brown', 'movie_reviews', 
    'reuters', 'conll2000', 'averaged_perceptron_tagger',
    'wordnet', 'vader_lexicon'
]

print("📥 NLTK veri setleri indiriliyor...")
for dataset in nltk_downloads:
    try:
        nltk.download(dataset, quiet=True)
        print(f"✅ {dataset} başarıyla indirildi")
    except Exception as e:
        print(f"❌ {dataset} indirilemedi: {e}")

print("\n🎉 Veri setleri hazır!")

# 🎲 Olasılıksal Dil Modellerine Giriş {#giris}

## 🤔 Neden Olasılıksal Modeller?

Doğal dil, karmaşık ve belirsizliklerle dolu bir yapıdır. Aynı kelimenin farklı anlamlarda kullanılması, cümle yapılarındaki çeşitlilik ve bağlamsal belirsizlikler, dil işlemede olasılıksal yaklaşımları zorunlu kılar.

### 📊 Temel Kavramlar

- **Dil Modeli**: Bir kelimenin veya cümlenin doğal bir dilde ne kadar olası olduğunu hesaplayan matematiksel model
- **Olasılık Dağılımı**: Dildeki olayların (kelimeler, etiketler) görülme sıklığını tanımlayan fonksiyon
- **Koşullu Olasılık**: Bir olayın, başka bir olay gerçekleştiğinde meydana gelme olasılığı

### 🎯 Uygulama Alanları

1. **Part-of-Speech Tagging**: Kelimelerin dilbilgisel türlerini belirleme
2. **Duygu Analizi**: Metinlerdeki duygusal tonun tespiti
3. **Makine Çevirisi**: Bir dilden diğerine çeviri
4. **Konuşma Tanıma**: Sesli verilerin metne dönüştürülmesi
5. **Metin Üretimi**: Otomatik metin oluşturma

In [ ]:
# Basit bir dil modeli örneği - Kelime frekansları
sample_text = """
Yapay zeka alanında doğal dil işleme çok önemli bir konudur. Doğal dil işleme, 
bilgisayarların insan dilini anlamasını ve işlemesini sağlar. Bu alanda makine 
öğrenmesi ve derin öğrenme teknikleri sıklıkla kullanılır. Doğal dil işleme 
uygulamaları arasında çeviri, duygu analizi ve metin özetleme yer alır.
"""

# Metni tokenize etme
tokens = word_tokenize(sample_text.lower())
tokens = [token for token in tokens if token.isalpha()]  # Sadece alfabetik karakterler

# Kelime frekanslarını hesaplama
word_freq = FreqDist(tokens)

# En sık kullanılan 10 kelimeyi görselleştirme
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
most_common = word_freq.most_common(10)
words, freqs = zip(*most_common)
bars = ax1.bar(words, freqs, color=pastel_colors[:len(words)])
ax1.set_title('En Sık Kullanılan 10 Kelime', fontsize=16, fontweight='bold')
ax1.set_xlabel('Kelimeler', fontsize=14)
ax1.set_ylabel('Frekans', fontsize=14)
ax1.tick_params(axis='x', rotation=45)

# Pie chart
ax2.pie(freqs, labels=words, autopct='%1.1f%%', colors=pastel_colors[:len(words)])
ax2.set_title('Kelime Dağılımı', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"📊 Toplam kelime sayısı: {len(tokens)}")
print(f"📚 Benzersiz kelime sayısı: {len(word_freq)}")
print(f"🔤 En sık kullanılan kelime: '{most_common[0][0]}' ({most_common[0][1]} kez)")

# 🔗 Hidden Markov Models (HMM) {#hmm}

## 🧠 Temel Kavramlar {#hmm-temel}

Hidden Markov Model (HMM), gözlemlenebilir olayların arkasında gizli durumların bulunduğu varsayımına dayanan olasılıksal bir modeldir. NLP'de özellikle Part-of-Speech (POS) tagging için yaygın olarak kullanılır.

### 📐 Matematiksel Temeller

HMM üç temel bileşenden oluşur:

1. **Durum Geçiş Olasılıkları (A)**: P(s_t | s_{t-1})
2. **Gözlem Olasılıkları (B)**: P(o_t | s_t)  
3. **Başlangıç Durumu Olasılıkları (π)**: P(s_1)

### 🎯 POS Tagging Örneği

Cümlede her kelime bir **gözlem**, her POS etiketi ise bir **gizli durum**dur.

In [ ]:
# HMM için basit bir görselleştirme
import matplotlib.patches as patches

fig, ax = plt.subplots(1, 1, figsize=(14, 8))

# Durum ve gözlem düğümleri
states = ['NOUN', 'VERB', 'ADJ', 'DET']
observations = ['cat', 'runs', 'fast', 'the']

# Durum düğümlerini çizme (gizli durumlar)
state_positions = [(2, 6), (5, 6), (8, 6), (11, 6)]
for i, (state, pos) in enumerate(zip(states, state_positions)):
    circle = plt.Circle(pos, 0.8, color=pastel_colors[i], alpha=0.7)
    ax.add_patch(circle)
    ax.text(pos[0], pos[1], state, ha='center', va='center', fontsize=12, fontweight='bold')

# Gözlem düğümlerini çizme
obs_positions = [(2, 3), (5, 3), (8, 3), (11, 3)]
for i, (obs, pos) in enumerate(zip(observations, obs_positions)):
    rect = patches.Rectangle((pos[0]-0.8, pos[1]-0.5), 1.6, 1, 
                           color=pastel_colors[i+4], alpha=0.7)
    ax.add_patch(rect)
    ax.text(pos[0], pos[1], obs, ha='center', va='center', fontsize=12, fontweight='bold')

# Durum geçişlerini çizme (oklar)
for i in range(len(state_positions)-1):
    start = state_positions[i]
    end = state_positions[i+1]
    ax.annotate('', xy=(end[0]-0.8, end[1]), xytext=(start[0]+0.8, start[1]),
                arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

# Gözlem bağlantılarını çizme
for i in range(len(state_positions)):
    start = state_positions[i]
    end = obs_positions[i]
    ax.annotate('', xy=(end[0], end[1]+0.5), xytext=(start[0], start[1]-0.8),
                arrowprops=dict(arrowstyle='->', lw=2, color='blue', alpha=0.6))

ax.set_xlim(0, 13)
ax.set_ylim(1, 8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Hidden Markov Model Yapısı\n(Gizli Durumlar: POS Etiketleri, Gözlemler: Kelimeler)', 
             fontsize=16, fontweight='bold', pad=20)

# Açıklama
ax.text(6.5, 1.5, 'Mavi oklar: Gözlem olasılıkları\nGri oklar: Durum geçiş olasılıkları', 
        ha='center', va='center', fontsize=12, 
        bbox=dict(boxstyle="round,pad=0.3", facecolor='lightblue', alpha=0.5))

plt.tight_layout()
plt.show()

## 🏷️ POS Tagging ile Uygulama {#hmm-pos}

Şimdi gerçek veri seti kullanarak HMM tabanlı POS tagger geliştireceğiz. Bu örnekte Brown Corpus'u kullanacağız.

In [ ]:
# Brown Corpus'tan veri hazırlığı
print("📚 Brown Corpus yükleniyor...")

# Brown corpus'tan tagged cümleler alıyoruz
brown_tagged_sents = brown.tagged_sents(categories='news')[:5000]  # İlk 5000 cümle

print(f"✅ {len(brown_tagged_sents)} cümle yüklendi")
print(f"📊 Örnek cümle: {brown_tagged_sents[0][:10]}")

# Veriyi train/test olarak ayırma
train_size = int(0.8 * len(brown_tagged_sents))
train_data = brown_tagged_sents[:train_size]
test_data = brown_tagged_sents[train_size:]

print(f"🏋️ Eğitim seti: {len(train_data)} cümle")
print(f"🧪 Test seti: {len(test_data)} cümle")

# POS etiketlerinin dağılımını analiz etme
all_tags = []
all_words = []

for sentence in train_data:
    for word, tag in sentence:
        all_tags.append(tag)
        all_words.append(word.lower())

tag_freq = FreqDist(all_tags)
word_freq = FreqDist(all_words)

print(f"🏷️ Toplam benzersiz POS etiketi: {len(tag_freq)}")
print(f"📚 Toplam benzersiz kelime: {len(word_freq)}")

In [ ]:
# POS etiketlerinin dağılımını görselleştirme
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# En sık kullanılan 15 POS etiketi
most_common_tags = tag_freq.most_common(15)
tags, freqs = zip(*most_common_tags)

ax1.barh(range(len(tags)), freqs, color=pastel_colors[:len(tags)])
ax1.set_yticks(range(len(tags)))
ax1.set_yticklabels(tags)
ax1.set_xlabel('Frekans', fontsize=14)
ax1.set_title('En Sık Kullanılan POS Etiketleri', fontsize=16, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Kelime uzunluklarının dağılımı
word_lengths = [len(word) for word in all_words]
ax2.hist(word_lengths, bins=20, color=pastel_colors[0], alpha=0.7, edgecolor='black')
ax2.set_xlabel('Kelime Uzunluğu', fontsize=14)
ax2.set_ylabel('Frekans', fontsize=14)
ax2.set_title('Kelime Uzunluklarının Dağılımı', fontsize=16, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# İstatistikler
print(f"📊 En sık kullanılan POS etiketi: {most_common_tags[0][0]} ({most_common_tags[0][1]:,} kez)")
print(f"📏 Ortalama kelime uzunluğu: {np.mean(word_lengths):.2f} karakter")
print(f"📐 Kelime uzunluğu std sapması: {np.std(word_lengths):.2f}")

In [ ]:
# HMM Tagger eğitimi
print("🏋️ HMM POS Tagger eğitiliyor...")

# HMM trainer oluşturma
trainer = hmm.HiddenMarkovModelTrainer()

# Modeli eğitme
hmm_tagger = trainer.train(train_data)

print("✅ HMM modeli başarıyla eğitildi!")

# Test cümlesi ile deneme
test_sentences = [
    "The quick brown fox jumps over the lazy dog".split(),
    "I love learning natural language processing".split(),
    "Machine learning algorithms are very powerful".split(),
    "Python is an excellent programming language".split()
]

print("\n🧪 Test Sonuçları:")
print("="*60)

for i, sentence in enumerate(test_sentences, 1):
    tagged = hmm_tagger.tag(sentence)
    print(f"\n{i}. Cümle: {' '.join(sentence)}")
    print(f"   Etiketler: {tagged}")
    
    # Etiketleri görselleştirme
    words, tags = zip(*tagged)
    print(f"   Formatlanmış: ", end="")
    for word, tag in tagged:
        print(f"{word}/{tag}", end=" ")
    print()

In [ ]:
# Modelin performansını değerlendirme
def evaluate_hmm_tagger(tagger, test_data_subset):
    """HMM tagger'ın doğruluğunu hesaplar"""
    correct = 0
    total = 0
    
    for sentence in test_data_subset:
        words = [word for word, tag in sentence]
        true_tags = [tag for word, tag in sentence]
        predicted_tags = [tag for word, tag in tagger.tag(words)]
        
        for true_tag, pred_tag in zip(true_tags, predicted_tags):
            if true_tag == pred_tag:
                correct += 1
            total += 1
    
    return correct / total if total > 0 else 0

# Test verisi alt kümesi ile değerlendirme (hesaplama süresini azaltmak için)
test_subset = test_data[:100]
accuracy = evaluate_hmm_tagger(hmm_tagger, test_subset)

print(f"🎯 HMM Tagger Doğruluğu: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Karışıklık matrisi için etiket analizi
true_tags_all = []
pred_tags_all = []

for sentence in test_subset[:20]:  # İlk 20 cümle
    words = [word for word, tag in sentence]
    true_tags = [tag for word, tag in sentence]
    predicted_tags = [tag for word, tag in tagger.tag(words)]
    
    true_tags_all.extend(true_tags)
    pred_tags_all.extend(predicted_tags)

# En sık karıştırılan etiketleri bulma
from collections import defaultdict
confusion_dict = defaultdict(int)

for true_tag, pred_tag in zip(true_tags_all, pred_tags_all):
    if true_tag != pred_tag:
        confusion_dict[(true_tag, pred_tag)] += 1

print("\n❌ En Sık Karıştırılan Etiket Çiftleri:")
for (true_tag, pred_tag), count in sorted(confusion_dict.items(), 
                                         key=lambda x: x[1], reverse=True)[:10]:
    print(f"   {true_tag} → {pred_tag}: {count} kez")

# ⚖️ Maximum Entropy Models {#maxent}

## 📖 Teori ve Matematiksel Temeller {#maxent-teori}

Maximum Entropy (MaxEnt) modelleri, verilen kısıtlar altında entropiyi maksimize eden olasılık dağılımlarını bulan güçlü sınıflandırma algoritmalarıdır. 

### 🧮 Matematiksel Formülasyon

MaxEnt modeli şu şekilde formüle edilir:

$$P(y|x) = \frac{1}{Z(x)} \exp\left(\sum_{i} \lambda_i f_i(x,y)\right)$$

Burada:
- $f_i(x,y)$: Özellik fonksiyonları
- $\lambda_i$: Özellik ağırlıkları  
- $Z(x)$: Normalizasyon sabiti

### 🎯 Avantajları

1. **Özellik Esnekliği**: Karmaşık özellikler tanımlanabilir
2. **Regularizasyon**: Overfitting'i önler
3. **Yorumlanabilirlik**: Özellik ağırlıkları anlamlıdır
4. **Performans**: Yüksek doğruluk oranları

## 💝 Duygu Analizi Uygulaması {#maxent-duygu}

Movie Reviews corpus'u kullanarak gelişmiş bir duygu analizi sistemi oluşturacağız.

In [ ]:
# Movie Reviews veri setini yükleme ve hazırlama
print("🎬 Movie Reviews veri seti yükleniyor...")

# Pozitif ve negatif yorumları alma
positive_reviews = [(list(movie_reviews.words(fileid)), 'positive') 
                   for fileid in movie_reviews.fileids('pos')]
negative_reviews = [(list(movie_reviews.words(fileid)), 'negative') 
                   for fileid in movie_reviews.fileids('neg')]

# Tüm yorumları birleştirme
all_reviews = positive_reviews + negative_reviews
random.shuffle(all_reviews)  # Rastgele karıştırma

print(f"✅ Toplam {len(all_reviews)} yorum yüklendi")
print(f"📈 Pozitif yorumlar: {len(positive_reviews)}")
print(f"📉 Negatif yorumlar: {len(negative_reviews)}")

# Veri setinin ilk birkaç örneğini gösterme
print(f"\n📄 Örnek pozitif yorum (ilk 20 kelime):")
print(" ".join(positive_reviews[0][0][:20]))
print(f"\n📄 Örnek negatif yorum (ilk 20 kelime):")
print(" ".join(negative_reviews[0][0][:20]))

# Yorum uzunluklarının analizi
review_lengths = [len(review[0]) for review in all_reviews]
print(f"\n📊 Ortalama yorum uzunluğu: {np.mean(review_lengths):.0f} kelime")
print(f"📊 Medyan yorum uzunluğu: {np.median(review_lengths):.0f} kelime")
print(f"📊 En uzun yorum: {max(review_lengths)} kelime")
print(f"📊 En kısa yorum: {min(review_lengths)} kelime")

In [ ]:
# Yorum uzunluklarını görselleştirme
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
ax1.hist(review_lengths, bins=50, color=pastel_colors[0], alpha=0.7, edgecolor='black')
ax1.set_xlabel('Yorum Uzunluğu (Kelime Sayısı)', fontsize=14)
ax1.set_ylabel('Frekans', fontsize=14)
ax1.set_title('Yorum Uzunluklarının Dağılımı', fontsize=16, fontweight='bold')
ax1.axvline(np.mean(review_lengths), color='red', linestyle='--', 
           label=f'Ortalama: {np.mean(review_lengths):.0f}')
ax1.axvline(np.median(review_lengths), color='blue', linestyle='--', 
           label=f'Medyan: {np.median(review_lengths):.0f}')
ax1.legend()
ax1.grid(alpha=0.3)

# Box plot pozitif vs negatif
pos_lengths = [len(review[0]) for review in positive_reviews]
neg_lengths = [len(review[0]) for review in negative_reviews]

ax2.boxplot([pos_lengths, neg_lengths], labels=['Pozitif', 'Negatif'], 
           patch_artist=True, 
           boxprops=dict(facecolor=pastel_colors[1], alpha=0.7),
           medianprops=dict(color='red', linewidth=2))
ax2.set_ylabel('Yorum Uzunluğu (Kelime Sayısı)', fontsize=14)
ax2.set_title('Pozitif vs Negatif Yorumların Uzunluk Karşılaştırması', 
             fontsize=16, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📊 Pozitif yorumların ortalama uzunluğu: {np.mean(pos_lengths):.0f} kelime")
print(f"📊 Negatif yorumların ortalama uzunluğu: {np.mean(neg_lengths):.0f} kelime")

In [ ]:
# Özellik çıkarma fonksiyonları
def extract_features(words):
    """Gelişmiş özellik çıkarma fonksiyonu"""
    features = {}
    
    # Temel kelime özellikleri
    word_set = set(words)
    features.update({f'contains({word})': True for word in word_set})
    
    # İstatistiksel özellikler
    features['word_count'] = len(words)
    features['unique_word_count'] = len(word_set)
    features['avg_word_length'] = np.mean([len(word) for word in words])
    
    # Duygu belirten kelimeler (basit sözlük tabanlı)
    positive_words = {'good', 'great', 'excellent', 'amazing', 'wonderful', 
                     'fantastic', 'perfect', 'love', 'best', 'brilliant'}
    negative_words = {'bad', 'terrible', 'awful', 'horrible', 'worst', 
                     'hate', 'stupid', 'boring', 'waste', 'disappointed'}
    
    features['positive_word_count'] = sum(1 for word in words if word.lower() in positive_words)
    features['negative_word_count'] = sum(1 for word in words if word.lower() in negative_words)
    
    # Büyük harf kullanımı (vurgu için)
    features['caps_count'] = sum(1 for word in words if word.isupper() and len(word) > 2)
    
    # Noktalama işaretleri
    features['exclamation_count'] = sum(1 for word in words if '!' in word)
    features['question_count'] = sum(1 for word in words if '?' in word)
    
    return features

# Tüm yorumlar için özellik çıkarma
print("🔧 Özellik çıkarma işlemi başlıyor...")
feature_sets = [(extract_features(words), label) for words, label in all_reviews]

# Veriyi train/test olarak ayırma
train_size = int(0.8 * len(feature_sets))
train_set = feature_sets[:train_size]
test_set = feature_sets[train_size:]

print(f"✅ {len(feature_sets)} yorum için özellikler çıkarıldı")
print(f"🏋️ Eğitim seti: {len(train_set)} yorum")
print(f"🧪 Test seti: {len(test_set)} yorum")

# Örnek özellikleri gösterme
sample_features = train_set[0][0]
print(f"\n📋 Örnek özellikler (ilk 10):")
feature_items = list(sample_features.items())[:10]
for key, value in feature_items:
    print(f"   {key}: {value}")

In [ ]:
# Maximum Entropy Classifier eğitimi
print("⚖️ Maximum Entropy Classifier eğitiliyor...")

# Modeli eğitme
maxent_classifier = MaxentClassifier.train(train_set, max_iter=20, trace=0)

print("✅ MaxEnt modeli başarıyla eğitildi!")

# Test seti üzerinde doğruluk hesaplama
test_accuracy = accuracy(maxent_classifier, test_set)
print(f"🎯 Test Doğruluğu: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Örnek tahminler
test_sentences = [
    "This movie is absolutely fantastic and I loved every minute of it",
    "Terrible movie, waste of time and money, very disappointing",
    "The film was okay, not great but not bad either",
    "Amazing acting and brilliant storytelling, highly recommended",
    "Boring plot with poor character development"
]

print(f"\n🧪 Örnek Tahminler:")
print("="*70)

for i, sentence in enumerate(test_sentences, 1):
    words = word_tokenize(sentence.lower())
    features = extract_features(words)
    prediction = maxent_classifier.classify(features)
    confidence = maxent_classifier.prob_classify(features)
    
    print(f"\n{i}. Cümle: \"{sentence}\"")
    print(f"   Tahmin: {prediction}")
    print(f"   Güven Oranları: Pozitif: {confidence.prob('positive'):.3f}, "
          f"Negatif: {confidence.prob('negative'):.3f}")

In [ ]:
# Model performansının detaylı analizi
from sklearn.metrics import classification_report

# Test seti tahminleri
y_true = [label for features, label in test_set]
y_pred = [maxent_classifier.classify(features) for features, label in test_set]

# Classification report
print("📊 Detaylı Performans Raporu:")
print("="*50)
print(classification_report(y_true, y_pred, target_names=['negative', 'positive']))

# Confusion Matrix görselleştirme
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred, labels=['negative', 'positive'])

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues', alpha=0.8)

# Metin ekleme
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=20, fontweight='bold')

ax.set_xticks(range(2))
ax.set_yticks(range(2))
ax.set_xticklabels(['Negative', 'Positive'], fontsize=14)
ax.set_yticklabels(['Negative', 'Positive'], fontsize=14)
ax.set_xlabel('Tahmin Edilen', fontsize=14)
ax.set_ylabel('Gerçek', fontsize=14)
ax.set_title('Confusion Matrix - MaxEnt Sınıflandırıcı', fontsize=16, fontweight='bold')

# Colorbar
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Güven aralıklarının dağılımı
confidences_positive = []
confidences_negative = []

for features, true_label in test_set[:200]:  # İlk 200 test örneği
    prob_dist = maxent_classifier.prob_classify(features)
    if true_label == 'positive':
        confidences_positive.append(prob_dist.prob('positive'))
    else:
        confidences_negative.append(prob_dist.prob('negative'))

# Güven dağılımlarını görselleştirme
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

ax.hist(confidences_positive, bins=20, alpha=0.7, label='Pozitif Örnekler', 
        color=pastel_colors[1], edgecolor='black')
ax.hist(confidences_negative, bins=20, alpha=0.7, label='Negatif Örnekler', 
        color=pastel_colors[0], edgecolor='black')

ax.set_xlabel('Güven Skoru', fontsize=14)
ax.set_ylabel('Frekans', fontsize=14)
ax.set_title('Model Güven Skorlarının Dağılımı', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"🎯 Pozitif örneklerde ortalama güven: {np.mean(confidences_positive):.3f}")
print(f"🎯 Negatif örneklerde ortalama güven: {np.mean(confidences_negative):.3f}")

# 📊 N-gram Dil Modelleri {#ngram}

## 🔤 Unigram, Bigram, Trigram {#ngram-temel}

N-gram modelleri, bir kelimenin önceki n-1 kelimeye bağlı olarak tahmin edildiği dil modelleridir. Dil modelleme ve metin üretimi için temel yapı taşlarıdır.

### 📐 Matematiksel Formülasyon

- **Unigram**: P(w_i)
- **Bigram**: P(w_i | w_{i-1})  
- **Trigram**: P(w_i | w_{i-2}, w_{i-1})

### 🎯 Kullanım Alanları

1. **Dil Modelleme**: Metin olasılıklarını hesaplama
2. **Metin Üretimi**: Yeni cümleler oluşturma
3. **Otomatik Tamamlama**: Kelime önerileri
4. **Makine Çevirisi**: Çeviri kalitesini artırma

In [ ]:
# Geniş bir corpus ile N-gram modeli oluşturma
print("📚 Reuters corpus'tan veri yükleniyor...")

# Reuters corpus'tan ilk 1000 makaleyi alıyoruz
reuters_sents = []
file_ids = reuters.fileids()[:1000]  # İlk 1000 dosya

for file_id in file_ids:
    words = reuters.words(file_id)
    # Sadece alfabetik karakterleri al ve küçük harfe çevir
    clean_words = [word.lower() for word in words if word.isalpha()]
    if len(clean_words) > 10:  # En az 10 kelime olan cümleleri al
        reuters_sents.append(clean_words)

# Tüm kelimeleri birleştir
all_words = [word for sent in reuters_sents for word in sent]

print(f"✅ {len(reuters_sents)} makale işlendi")
print(f"📊 Toplam kelime sayısı: {len(all_words):,}")
print(f"📚 Benzersiz kelime sayısı: {len(set(all_words)):,}")

# En sık kullanılan kelimeleri göster
word_freq = FreqDist(all_words)
most_common = word_freq.most_common(20)

print(f"\n🔝 En sık kullanılan 20 kelime:")
for i, (word, freq) in enumerate(most_common, 1):
    print(f"{i:2d}. {word:12s} ({freq:,} kez)")

In [ ]:
# N-gram'ları oluşturma ve analiz etme
print("🔧 N-gram'lar oluşturuluyor...")

# Unigrams (1-gram)
unigrams = list(ngrams(all_words, 1))
unigram_freq = FreqDist(unigrams)

# Bigrams (2-gram)  
bigrams = list(ngrams(all_words, 2))
bigram_freq = FreqDist(bigrams)

# Trigrams (3-gram)
trigrams = list(ngrams(all_words, 3))
trigram_freq = FreqDist(trigrams)

# 4-grams
fourgrams = list(ngrams(all_words, 4))
fourgram_freq = FreqDist(fourgrams)

print(f"✅ N-gram'lar oluşturuldu!")
print(f"📊 Unigram sayısı: {len(unigram_freq):,}")
print(f"📊 Bigram sayısı: {len(bigram_freq):,}")
print(f"📊 Trigram sayısı: {len(trigram_freq):,}")
print(f"📊 4-gram sayısı: {len(fourgram_freq):,}")

# N-gram istatistiklerini görselleştirme
ngram_counts = [len(unigram_freq), len(bigram_freq), len(trigram_freq), len(fourgram_freq)]
ngram_labels = ['Unigram', 'Bigram', 'Trigram', '4-gram']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
bars = ax1.bar(ngram_labels, ngram_counts, color=pastel_colors[:4])
ax1.set_ylabel('Benzersiz N-gram Sayısı', fontsize=14)
ax1.set_title('N-gram Çeşitliliği', fontsize=16, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Değerleri bar'ların üzerine yazma
for bar, count in zip(bars, ngram_counts):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + max(ngram_counts)*0.01,
             f'{count:,}', ha='center', va='bottom', fontweight='bold')

# En sık bigram'ları görselleştirme
most_common_bigrams = bigram_freq.most_common(15)
bigram_words = [f"{bg[0]}-{bg[1]}" for bg, freq in most_common_bigrams]
bigram_freqs = [freq for bg, freq in most_common_bigrams]

ax2.barh(range(len(bigram_words)), bigram_freqs, color=pastel_colors[1])
ax2.set_yticks(range(len(bigram_words)))
ax2.set_yticklabels(bigram_words, fontsize=10)
ax2.set_xlabel('Frekans', fontsize=14)
ax2.set_title('En Sık Kullanılan 15 Bigram', fontsize=16, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# En sık kullanılan trigram'ları göster
print(f"\n🔝 En sık kullanılan 10 trigram:")
for i, (trigram, freq) in enumerate(trigram_freq.most_common(10), 1):
    trigram_str = "-".join(trigram)
    print(f"{i:2d}. {trigram_str:25s} ({freq} kez)")

## 🎭 Dil Modelleme ve Metin Üretimi {#ngram-uretim}

N-gram modellerini kullanarak metin üretimi yapacağız.

In [ ]:
# Metin üretimi için N-gram tabanlı dil modeli
class NgramLanguageModel:
    def __init__(self, n=2):
        self.n = n
        self.ngrams = defaultdict(list)
        
    def train(self, sentences):
        """N-gram modelini eğitir"""
        for sentence in sentences:
            # Cümlenin başına ve sonuna özel tokenlar ekle
            padded_sentence = ['<START>'] * (self.n - 1) + sentence + ['<END>']
            
            # N-gram'ları oluştur
            for i in range(len(padded_sentence) - self.n + 1):
                ngram = tuple(padded_sentence[i:i + self.n])
                context = ngram[:-1]
                next_word = ngram[-1]
                self.ngrams[context].append(next_word)
    
    def predict_next_word(self, context):
        """Verilen bağlam için sonraki kelimeyi tahmin eder"""
        context = tuple(context[-(self.n-1):])  # Son n-1 kelimeyi al
        if context in self.ngrams:
            next_words = self.ngrams[context]
            # Frekansa göre rastgele seçim
            return random.choice(next_words)
        else:
            # Eğer bağlam bulunamazsa, en sık kullanılan kelimeyi döndür
            all_words = [word for words in self.ngrams.values() for word in words]
            if all_words:
                return random.choice(all_words)
            return '<UNK>'
    
    def generate_text(self, start_words=None, max_length=20):
        """Metin üretir"""
        if start_words is None:
            start_words = ['<START>'] * (self.n - 1)
        else:
            start_words = ['<START>'] * (self.n - 1) + start_words
            
        generated = start_words[:]
        
        for _ in range(max_length):
            next_word = self.predict_next_word(generated[-(self.n-1):])
            if next_word == '<END>':
                break
            generated.append(next_word)
        
        # Başlangıç tokenlarını çıkar
        return generated[(self.n-1):]

# Farklı N değerleri için modeller oluşturma
models = {}
for n in [2, 3, 4]:
    print(f"🔧 {n}-gram modeli eğitiliyor...")
    model = NgramLanguageModel(n=n)
    model.train(reuters_sents[:500])  # İlk 500 cümle ile eğit
    models[n] = model

print("✅ Tüm modeller eğitildi!")

In [ ]:
# Farklı modeller ile metin üretimi
start_words_list = [
    ['the', 'company'],
    ['market', 'prices'],
    ['economic', 'growth'],
    ['government', 'policy']
]

print("🎭 Metin Üretimi Örnekleri")
print("="*80)

for start_words in start_words_list:
    print(f"\n🌱 Başlangıç kelimeleri: {' '.join(start_words)}")
    print("-" * 60)
    
    for n in [2, 3, 4]:
        generated = models[n].generate_text(start_words=start_words, max_length=15)
        # <END> tokenını çıkar
        generated = [word for word in generated if word != '<END>']
        generated_text = ' '.join(generated)
        print(f"{n}-gram: {generated_text}")

# Rastgele metin üretimi
print(f"\n\n🎲 Rastgele Metin Üretimleri")
print("="*80)

for i in range(5):
    print(f"\n{i+1}. Örnek:")
    for n in [2, 3, 4]:
        generated = models[n].generate_text(max_length=12)
        generated = [word for word in generated if word != '<END>']
        generated_text = ' '.join(generated)
        print(f"   {n}-gram: {generated_text}")

# 🚀 5 Kapsamlı Case Study Projeleri {#projeler}

Bu bölümde öğrendiklerimizi pekiştirmek için 5 farklı gerçek dünya projesi geliştirilmiştir. Her proje ayrı bir notebook dosyası olarak sunulmaktadır ve 4000+ veri örneği içerecek şekilde tasarlanmıştır.

## 📁 Proje Dosyaları

1. **📰 [Case Study 1: Reuters Haber Kategorisi Sınıflandırma](./case_study_1_news_classification.ipynb)**
   - Maximum Entropy Models kullanarak haber kategorilerini sınıflandırma
   - 10 farklı kategori (acq, corn, crude, earn, grain, interest, money-fx, ship, trade, wheat)
   - 4000+ haber metni
   - Finansal ve tarımsal terim analizi

2. **🏷️ [Case Study 2: Gelişmiş POS Tagging Sistemi](./case_study_2_pos_tagging.ipynb)**
   - Hidden Markov Models ile part-of-speech tagging
   - ConLL-2000 veri seti kullanımı
   - 4000+ etiketli cümle
   - Hata analizi ve görselleştirme

3. **📚 [Case Study 3: Büyük Ölçekli Dil Modeli](./case_study_3_language_model.ipynb)**
   - N-gram tabanlı dil modelleme ve metin üretimi
   - Reuters corpus ile eğitim
   - Smoothing teknikleri ve perplexity analizi
   - Otomatik metin üretimi

4. **🎬 [Case Study 4: Çok Sınıflı Duygu Analizi](./case_study_4_sentiment_analysis.ipynb)**
   - 5 seviyeli duygu sınıflandırma (very negative → very positive)
   - Maximum Entropy ile çok sınıflı classification
   - Gelişmiş özellik mühendisliği
   - Güven skoru analizi

5. **🔄 [Case Study 5: Hibrit NLP Sistemi](./case_study_5_hybrid_system.ipynb)**
   - HMM, MaxEnt ve N-gram modellerinin kombinasyonu
   - Pipeline tabanlı metin işleme
   - Gerçek zamanlı analiz sistemi
   - Performans karşılaştırması

## 🎯 Her Proje İçeriği

- **Veri Hazırlığı**: Büyük ölçekli veri setleri
- **Model Geliştirme**: Adım adım implementasyon
- **Görselleştirme**: Profesyonel grafikler ve analizler
- **Değerlendirme**: Detaylı performans metrikleri
- **Gerçek Örnekler**: Pratik kullanım senaryoları

## 📋 Çalıştırma Sırası

1. Bu ana notebook'u tamamladıktan sonra
2. Her case study'yi sırayla çalıştırın
3. Her projede önceki konuları pekiştirin
4. Son projede tüm teknikleri birleştirin

# 📈 Sonuçlar ve Değerlendirme {#sonuc}

## 🎓 Öğrenilen Konular

Bu notebook serisinde aşağıdaki konuları detaylı olarak öğrendiniz:

### 🔗 Hidden Markov Models (HMM)
- **Temel Kavramlar**: Gizli durumlar, geçiş olasılıkları, gözlem olasılıkları
- **POS Tagging**: Kelimelerin dilbilgisel türlerini belirleme
- **Viterbi Algoritması**: En olası durum dizisini bulma
- **Praktik Uygulamalar**: Gerçek corpus'lar ile model eğitimi

### ⚖️ Maximum Entropy Models
- **Matematiksel Temeller**: Entropi maksimizasyonu ve regularizasyon
- **Özellik Mühendisliği**: Etkili özellik çıkarma teknikleri
- **Sınıflandırma**: İkili ve çok sınıflı problemler
- **Duygu Analizi**: Metin tabanlı duygu tespiti

### 📊 N-gram Dil Modelleri
- **Dil Modelleme**: Kelimelerin olasılık dağılımları
- **Metin Üretimi**: Otomatik içerik oluşturma
- **Smoothing Teknikleri**: Sıfır olasılık probleminin çözümü
- **Perplexity**: Model kalitesinin değerlendirilmesi

## 🛠️ Teknik Beceriler

- **NLTK**: Doğal dil işleme kütüphanesi kullanımı
- **Corpus İşleme**: Büyük ölçekli veri setleri ile çalışma
- **Model Eğitimi**: Olasılıksal modellerin eğitimi ve optimizasyonu
- **Değerlendirme**: Accuracy, precision, recall, F1-score metrikleri
- **Görselleştirme**: Matplotlib ve Seaborn ile profesyonel grafikler

## 🚀 Sonraki Adımlar

1. **Derin Öğrenme**: Transformer modelleri (BERT, GPT)
2. **Word Embeddings**: Word2Vec, GloVe, FastText
3. **Sequence Models**: LSTM, GRU tabanlı modeller
4. **Advanced NLP**: Named Entity Recognition, Relation Extraction
5. **Multimodal**: Metin ve görsel verilerin birleşimi

## 📚 Kaynaklar

- **Kitaplar**:
  - "Speech and Language Processing" - Jurafsky & Martin
  - "Natural Language Processing with Python" - Bird, Klein & Loper
  - "Foundations of Statistical Natural Language Processing" - Manning & Schütze

- **Online Kurslar**:
  - CS224N: Natural Language Processing (Stanford)
  - Deep Learning Specialization (Coursera)
  - Fast.ai NLP Course

- **Araştırma Makaleleri**:
  - "A Maximum Entropy Approach to Natural Language Processing" - Berger et al.
  - "Statistical Methods for Speech Recognition" - Jelinek
  - "Empirical Methods in Natural Language Processing" - Conference Proceedings

## 🎯 Pratik Öneriler

1. **Farklı Veri Setleri**: Kendi veri setlerinizi deneyın
2. **Model Karşılaştırması**: Farklı algoritmaları karşılaştırın
3. **Hiperparametre Optimizasyonu**: Grid search ile en iyi parametreleri bulun
4. **Gerçek Projeler**: Kendi NLP projelerinizi geliştirin
5. **Topluluk**: NLP topluluklarına katılın ve projelerinizi paylaşın

---

**🎉 Tebrikler!** Olasılıksal Dil Modelleri konusunda sağlam bir temel oluşturdunuz. Şimdi case study projelerine geçerek öğrendiklerinizi pratikte uygulayabilirsiniz!